# Painel de Analise de Vendas
Notebook referente ao painel interativo de vendas do varejo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
sns.set_palette(sns.color_palette("deep"))
%matplotlib inline

## Etapa 1 - Preparacao e importacao
Execute o arquivo `vendas.csv` junto a este notebook no Colab.

In [ ]:
from pathlib import Path

caminho_csv = Path("vendas.csv")
if not caminho_csv.exists():
    try:
        from google.colab import files
        print("Selecione o arquivo vendas.csv.")
        enviado = files.upload()
        caminho_csv = Path(next(iter(enviado)))
    except ImportError:
        raise FileNotFoundError("O arquivo vendas.csv nao foi encontrado.")

df = pd.read_csv(caminho_csv)
df["data"] = pd.to_datetime(df["data"], errors="coerce")
df["faturamento"] = df["quantidade"] * df["preco_unitario"]
df["mes"] = df["data"].dt.month
df.head()

In [ ]:
print("Informacoes do dataset")
df.info()

print("\nValores ausentes")
print(df.isnull().sum())

print("\nResumo estatistico")
display(df.describe(include="all"))

In [ ]:
df["quantidade"] = df["quantidade"].fillna(df["quantidade"].mean())
df["preco_unitario"] = df["preco_unitario"].fillna(df["preco_unitario"].mean())
df["faturamento"] = df["quantidade"] * df["preco_unitario"]

print("Valores ausentes apos o tratamento:")
print(df.isnull().sum())

## Etapa 2 - Analise exploratoria

In [ ]:
faturamento_loja = df.groupby("loja", as_index=True)["faturamento"].sum().sort_values(ascending=False)
print("Faturamento total por loja")
display(faturamento_loja)

In [ ]:
ticket_medio = df.groupby("loja").agg(
    faturamento_total=("faturamento", "sum"),
    numero_vendas=("faturamento", "count")
)
ticket_medio["ticket_medio"] = ticket_medio["faturamento_total"] / ticket_medio["numero_vendas"]
ticket_medio = ticket_medio.sort_values("ticket_medio", ascending=False)
display(ticket_medio)

In [ ]:
cinco_mais_vendidos = df.groupby("produto", as_index=False)["quantidade"].sum().sort_values("quantidade", ascending=False).head(5)
display(cinco_mais_vendidos)

In [ ]:
resumo_mensal = df.assign(mes_ano=df["data"].dt.to_period("M")).groupby("mes_ano")["faturamento"].sum()
mes_maior = resumo_mensal.idxmax()
mes_menor = resumo_mensal.idxmin()
print(f"Mes com maior faturamento: {mes_maior.strftime('%m/%Y')} - R$ {resumo_mensal.max():,.2f}")
print(f"Mes com menor faturamento: {mes_menor.strftime('%m/%Y')} - R$ {resumo_mensal.min():,.2f}")

## Etapa 3 - Visualizacoes

In [ ]:
Path("graficos").mkdir(exist_ok=True)

cores = sns.color_palette("deep", n_colors=len(faturamento_loja))
ax = faturamento_loja.plot(kind="bar", color=cores, figsize=(10, 6))
ax.set_title("Faturamento Total por Loja")
ax.set_xlabel("Loja")
ax.set_ylabel("Faturamento (R$)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("graficos/faturamento_por_loja.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
ax = resumo_mensal.plot(marker="o", linewidth=2)
ax.set_title("Evolucao Mensal do Faturamento")
ax.set_xlabel("Mes/Ano")
ax.set_ylabel("Faturamento (R$)")
rotulos = [periodo.strftime("%m/%Y") for periodo in resumo_mensal.index]
plt.xticks(range(len(rotulos)), rotulos, rotation=45)
for x, valor in enumerate(resumo_mensal.values):
    ax.annotate(f"R$ {valor:,.0f}", (x, valor), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("graficos/evolucao_mensal_faturamento.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
faturamento_regiao = df.groupby("regiao")["faturamento"].sum().sort_values(ascending=False)
cores_pizza = sns.color_palette("deep", n_colors=len(faturamento_regiao))
plt.figure(figsize=(9, 7))
plt.pie(faturamento_regiao, labels=faturamento_regiao.index, autopct="%1.1f%%", startangle=90, colors=cores_pizza)
plt.title("Distribuicao do Faturamento por Regiao")
plt.axis("equal")
plt.tight_layout()
plt.savefig("graficos/faturamento_por_regiao.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x="quantidade", y="faturamento", hue="loja", alpha=0.75)
plt.title("Relacao entre Quantidade Vendida e Faturamento")
plt.xlabel("Quantidade vendida")
plt.ylabel("Faturamento (R$)")
plt.legend(title="Loja", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig("graficos/quantidade_vs_faturamento.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(11, 7))
sns.boxplot(data=df, x="categoria", y="preco_unitario", hue="categoria", palette="deep", legend=False)
plt.title("Distribuicao dos Precos Unitarios por Categoria")
plt.xlabel("Categoria")
plt.ylabel("Preco unitario (R$)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig("graficos/precos_por_categoria.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
correlacao = df.select_dtypes(include=[np.number]).corr()
plt.figure(figsize=(9, 7))
sns.heatmap(correlacao, annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5, square=True)
plt.title("Correlacao entre Variaveis Numericas")
plt.tight_layout()
plt.savefig("graficos/heatmap_correlacao.png", dpi=150, bbox_inches="tight")
plt.show()

## Etapa 4 - Apresentacao consolidada

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

faturamento_loja.plot(kind="bar", ax=axes[0], color=sns.color_palette("deep", n_colors=len(faturamento_loja)))
axes[0].set_title("Faturamento por Loja")
axes[0].set_xlabel("Loja")
axes[0].set_ylabel("Faturamento (R$)")
axes[0].tick_params(axis="x", rotation=25)

axes[1].pie(faturamento_regiao, labels=faturamento_regiao.index, autopct="%1.1f%%", startangle=90, colors=sns.color_palette("deep", n_colors=len(faturamento_regiao)))
axes[1].set_title("Faturamento por Regiao")
axes[1].axis("equal")

plt.tight_layout()
plt.savefig("graficos/comparativo_loja_regiao.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
resumo_lojas = df.groupby("loja").agg(
    faturamento_total=("faturamento", "sum"),
    quantidade_vendida=("quantidade", "sum"),
    numero_vendas=("faturamento", "count")
).reset_index()
resumo_lojas["ticket_medio"] = resumo_lojas["faturamento_total"] / resumo_lojas["numero_vendas"]
resumo_lojas = resumo_lojas.sort_values("faturamento_total", ascending=False)
resumo_lojas.to_csv("resumo_analitico.csv", index=False, encoding="utf-8-sig")
display(resumo_lojas)

## Observacoes finais
- A maior concentracao de faturamento pode ser comparada diretamente no grafico por loja.
- A evolucao mensal permite identificar crescimento, quedas e sazonalidade.
- A participacao regional mostra quais mercados possuem maior representatividade.
- O grafico de dispersao ajuda a observar a relacao entre volume vendido e faturamento, bem como possiveis valores extremos.
- O boxplot revela categorias com maior variacao de preco unitario.
- O heatmap resume correlacoes entre as variaveis numericas do conjunto de dados.